# 03. 0단계 점검 — 세탁만 그래프의 구조

유형 모델 상한선 실험은 **세탁 거래만 모은 그래프**(분할 끝까지 누적)를 입력으로 쓴다. 학습 전에 이 그래프가 어떻게 생겼는지 본다.
상한선 성능이 예상보다 높게 나올 때, 그게 구조를 배운 결과인지 쉬운 규칙 덕분인지 가를 근거가 여기서 나온다.

1. **이웃 종류** — 대상 엣지의 양끝 계좌에 붙은 다른 세탁 엣지가 없음 / UNMATCHED 만 / 패턴만 / 섞임.
   `given_kind` 에서 "isolated 면 거의 UNMATCHED" 처럼 한 클래스로 쏠리면 → 상한선의 UNMATCHED 성능은 쉬운 규칙. 이진 오탐이 섞이는 실제 조건(8단계)에서 크게 떨어질 것
2. **연결 요소** — 요소 하나 = 시도 하나인지, 여러 시도·UNMATCHED 가 엉켜 있는지, 거대 요소가 있는지
3. **계좌 공유** — 한 계좌가 여러 시도에 쓰이는지, 다른 유형 시도와 섞이는지

그래프 시점: 1 은 분할마다 그 분할 끝, 2·3 은 `CUTOFF`(기본 테스트 끝)

## 0. 설정

In [1]:
import sys
from pathlib import Path
import pandas as pd
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 220)

sys.path.append(str(Path.cwd()))
import mc_lib as M

DATASET = "HI-Large"
TRAIN = ("2022-08-08", "2022-10-02")
VAL   = ("2022-10-03", "2022-10-16")
TEST  = ("2022-10-17", "2022-10-30")
CUTOFF = TEST[1]                      # 연결 요소·계좌 공유를 볼 시점 (이 날짜까지 누적)

windows = M.make_windows(TRAIN, VAL, TEST)
OUT = M.results_dir(DATASET, "03_graph_check")
lau = M.load_laundering(DATASET)
print(f"세탁 엣지 {len(lau):,}")

세탁 엣지 225,546


## 1. 이웃 종류

`by_class` : 분할 × 클래스 → 이웃 종류 비율(%), `other_med` = 양끝 계좌의 다른 세탁 엣지 수 중앙값

In [2]:
nc = M.neighbor_context(lau, windows)
nc["by_class"]

kind                  isolated  only_unmatched  only_pattern  mixed  other_med      n
split pattern_type                                                                   
train FAN-OUT             0.73            0.12         82.98  16.18       11.0   6045
      FAN-IN              0.67            0.00         82.26  17.08       11.0   6465
      GATHER-SCATTER      0.45            0.06         83.53  15.96       15.0   9012
      SCATTER-GATHER      0.18            0.02         82.64  17.15       12.0  12537
      CYCLE               1.10            0.15         83.59  15.17        2.0   6025
      RANDOM              2.12            0.36         80.46  17.06        2.0   4948
      BIPARTITE          55.18           10.35         28.34   6.14        0.0   6437
      STACK               8.59            1.06         75.53  14.82        1.0  11420
      UNMATCHED          40.77           49.16          5.75   4.32        1.0  47782
val   FAN-OUT             2.37            0.56         80.71  16.36        8.0   1980
      FAN-IN              1.74            0.26         81.44  16.55        8.0   1897
      GATHER-SCATTER      0.78            0.10         79.14  19.97       13.0   3845
      SCATTER-GATHER      0.59            0.05         80.61  18.75       10.0   3909
      CYCLE               2.83            0.79         80.74  15.64        2.0   1765
      RANDOM              4.07            0.84         78.74  16.35        2.0   1425
      BIPARTITE          50.58            8.61         33.41   7.40        0.0   1730
      STACK              18.96            2.99         63.65  14.40        1.0   3549
      UNMATCHED          37.53           48.05          7.97   6.45        1.0  13695
test  FAN-OUT             1.27            0.33         79.59  18.82        8.0   2131
      FAN-IN              1.77            0.16         77.78  20.29        9.0   1922
      GATHER-SCATTER      0.85            0.21         77.60  21.35       14.0   3785
      SCATTER-GATHER      0.62            0.03         78.41  20.95       11.0   3891
      CYCLE               2.21            0.49         79.38  17.92        2.0   1853
      RANDOM              3.90            0.37         76.18  19.55        2.0   1335
      BIPARTITE          43.92            9.44         36.12  10.53        1.0   1833
      STACK              17.72            3.23         62.44  16.61        1.0   3437
      UNMATCHED          33.50           49.94          8.56   7.99        2.0  14273

`given_kind` : 분할 × 이웃 종류 → 클래스 분포(%), 행 합 100. 한 클래스로 쏠린 행이 쉬운 규칙

In [3]:
nc["given_kind"]

pattern_type          FAN-OUT  FAN-IN  GATHER-SCATTER  SCATTER-GATHER  CYCLE  RANDOM  BIPARTITE  STACK  UNMATCHED      n
split kind                                                                                                              
train isolated           0.18    0.18            0.17            0.09   0.27    0.43      14.60   4.03      80.05  24337
      only_unmatched     0.03    0.00            0.02            0.01   0.04    0.07       2.74   0.50      96.59  24317
      only_pattern       9.95   10.54           14.93           20.54   9.98    7.89       3.62  17.10       5.44  50436
      mixed              8.44    9.53           12.42           18.56   7.89    7.29       3.41  14.61      17.84  11581
val   isolated           0.68    0.48            0.43            0.33   0.72    0.84      12.63   9.71      74.18   6929
      only_unmatched     0.16    0.07            0.06            0.03   0.20    0.17       2.16   1.54      95.60   6883
      only_pattern      10.11    9.77           19.24           19.93   9.01    7.10       3.66  14.29       6.90  15812
      mixed              7.77    7.53           18.41           17.57   6.62    5.59       3.07  12.25      21.19   4171
test  isolated           0.42    0.53            0.50            0.37   0.64    0.81      12.57   9.51      74.65   6406
      only_unmatched     0.09    0.04            0.11            0.01   0.12    0.07       2.32   1.49      95.74   7445
      only_pattern      10.80    9.52           18.71           19.44   9.37    6.48       4.22  13.67       7.78  15697
      mixed              8.16    7.94           16.45           16.59   6.76    5.31       3.93  11.62      23.23   4912

## 2. 연결 요소 (방향 무시)

- `summary` : 요소 수, 요소 크기(엣지 수) 분위, 가장 큰 요소
- `kinds` : 요소 종류별 개수·엣지 비율 — `unmatched_only` / `single_attempt`(시도 1개만) / `single_attempt+unm` / `multi_attempt`
- `by_pattern` : 패턴별 시도 중 자기 요소에 다른 시도가 있는 비율, UNMATCHED 가 섞인 비율
- `by_class` : 클래스별 엣지가 어떤 종류 요소에 있나 (%)

In [4]:
cc = M.components(lau, CUTOFF)
cc["summary"]

cutoff                     2022-10-30
edges                          186132
nodes                          196430
components                      47515
comp_edges_p50                    1.0
comp_edges_p90                    3.0
comp_edges_p99                   13.0
largest_edges                   91813
largest_nodes                   73049
largest_attempts                 7943
largest_edges_share_pct         49.33
dtype: object

In [5]:
cc["kinds"]

,components,edges,edges_%
kind,,,
unmatched_only,34849,46922,25.21
single_attempt,9016,14781,7.94
single_attempt+unm,2092,8423,4.53
multi_attempt,1558,116006,62.32


In [6]:
cc["by_pattern"]

,attempts,other_attempt_in_comp_%,unmatched_in_comp_%,comp_edges_med
pattern_type,,,,
FAN-OUT,1906,87.83,68.36,91813.0
FAN-IN,1880,86.81,66.22,91813.0
GATHER-SCATTER,1929,91.50,67.39,91813.0
SCATTER-GATHER,1925,89.56,68.16,91813.0
CYCLE,1978,82.56,62.18,550.0
RANDOM,1878,82.85,62.67,550.0
BIPARTITE,1873,63.59,45.27,101.0
STACK,1866,70.26,51.98,165.0


In [7]:
cc["by_class"]

kind,unmatched_only,single_attempt,single_attempt+unm,multi_attempt
pattern_type,,,,
FAN-OUT,0.00,4.97,4.11,90.92
FAN-IN,0.00,6.93,3.32,89.74
GATHER-SCATTER,0.00,2.92,1.48,95.60
SCATTER-GATHER,0.00,3.79,4.30,91.91
CYCLE,0.00,9.62,4.68,85.70
RANDOM,0.00,9.23,5.50,85.27
BIPARTITE,0.00,43.33,7.93,48.74
STACK,0.00,33.25,8.81,57.94
UNMATCHED,57.81,0.00,3.91,38.28


In [8]:
# 큰 요소 상위 10개
cc["comp"].sort_values("n_edges", ascending=False).head(10)

,n_edges,n_attempts,n_pattern,n_unmatched,n_nodes,kind
comp,,,,,,
0,91813,7943,62537,29276,73049,multi_attempt
3792,1150,150,1150,0,17,multi_attempt
1119,816,116,813,3,13,multi_attempt
1168,623,187,622,1,8,multi_attempt
43756,550,294,550,0,3,multi_attempt
2241,499,0,0,499,442,unmatched_only
12286,499,154,499,0,5,multi_attempt
1634,405,0,0,405,370,unmatched_only
2286,397,0,0,397,370,unmatched_only


## 3. 계좌 공유

- `accounts` : 패턴 계좌별 참여 시도 수 (1 = 시도 하나에만)
- `by_pattern` : 패턴별 시도 중 다른 시도와 계좌 공유 / 다른 유형 시도와 공유 / UNMATCHED 엣지가 있는 계좌 포함 비율(%)

In [9]:
sa = M.shared_accounts(lau, CUTOFF)
sa["accounts"]

1     82902
2      8666
3      1125
4       280
5+      563
Name: accounts, dtype: int64

In [10]:
sa["by_pattern"]

,attempts,shared_%,shared_other_type_%,has_unmatched_account_%
pattern_type,,,,
FAN-OUT,1906,84.05,82.16,50.58
FAN-IN,1880,84.47,82.02,42.87
GATHER-SCATTER,1929,89.99,88.70,51.43
SCATTER-GATHER,1925,87.27,85.77,50.29
CYCLE,1978,78.82,77.45,40.65
RANDOM,1878,79.45,77.96,41.05
BIPARTITE,1873,89.70,87.67,58.09
STACK,1866,92.77,90.35,60.88


## 4. 저장

In [11]:
for name, df in {"neighbor_by_class": nc["by_class"], "neighbor_given_kind": nc["given_kind"],
                 "components_summary": cc["summary"], "components_kinds": cc["kinds"],
                 "components_by_pattern": cc["by_pattern"], "components_by_class": cc["by_class"],
                 "shared_accounts": sa["accounts"], "shared_by_pattern": sa["by_pattern"]}.items():
    df.to_csv(OUT / f"{name}.csv", encoding="utf-8-sig")
sorted(p.name for p in OUT.iterdir())

['components_by_class.csv',
 'components_by_pattern.csv',
 'components_kinds.csv',
 'components_summary.csv',
 'neighbor_by_class.csv',
 'neighbor_given_kind.csv',
 'shared_accounts.csv',
 'shared_by_pattern.csv']